In [ ]:
!pip install -q osmnx geopandas pyproj shapely fiona pyogrio

import os, math, time, random
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from shapely.ops import unary_union
import osmnx as ox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.5 MB/s eta 0:00:00


In [ ]:
#For the purpose of reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [ ]:
#Parse & deduplicate positive KML files
RAW_KML_FILES = [
    "all_warehouses_cleaned_0.kml",
    "all_warehouses_cleaned_2000.kml",
    "all_warehouses_cleaned_4000.kml",
]
CLEAN_DIR = "/content/kmls_cleaned"
os.makedirs(CLEAN_DIR, exist_ok=True)


def clean_kml_file(input_path: str, output_path: str) -> None:
    """Fix common XML/KML issues: BOM, leading whitespace, truncate to first valid tag."""
    with open(input_path, "r", encoding="utf-8-sig", errors="ignore") as f:
        text = f.read()
    text = text.lstrip()
    xml_idx, kml_idx = text.find("<?xml"), text.find("<kml")
    valid = [i for i in [xml_idx, kml_idx] if i != -1]
    if not valid:
        raise ValueError(f"No valid XML/KML start found in: {input_path}")
    text = text[min(valid):]
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)


def batch_clean_kmls(raw_files):
    cleaned_files = []
    for i, fp in enumerate(raw_files):
        out_fp = os.path.join(CLEAN_DIR, f"warehouses_cleaned_{i}.kml")
        clean_kml_file(fp, out_fp)
        cleaned_files.append(out_fp)
        print(f"[clean] {fp} -> {out_fp}")
    return cleaned_files


def load_positive_points(cleaned_kml_files):
    gdf_list = []
    for fp in cleaned_kml_files:
        gdf_temp = gpd.read_file(fp, driver="KML")
        gdf_temp["source_file"] = Path(fp).name
        gdf_list.append(gdf_temp)
        print(f"[load] {fp}: rows = {len(gdf_temp)}")

    pos = pd.concat(gdf_list, ignore_index=True)
    pos = gpd.GeoDataFrame(pos, geometry="geometry", crs=gdf_list[0].crs)
    pos = pos[pos.geometry.notnull() & (pos.geometry.geom_type == "Point")].copy()
    pos["lon"], pos["lat"] = pos.geometry.x, pos.geometry.y
    pos = pos.dropna(subset=["lat", "lon"]).drop_duplicates(subset=["lat", "lon"]).reset_index(drop=True)
    pos["pos_id"] = pos.index.astype(str).str.zfill(7)
    print(f"[pos] unique cleaned point count = {len(pos)}")
    return pos


cleaned_kml_files = batch_clean_kmls(RAW_KML_FILES)
pos_gdf = load_positive_points(cleaned_kml_files)
pos_gdf[["pos_id", "lat", "lon", "source_file"]].to_csv(
    os.path.join(CLEAN_DIR, "positive_points_cleaned_dedup.csv"), index=False
)
print(pos_gdf.shape)


[clean] all_warehouses_cleaned_0.kml -> /content/kmls_cleaned/warehouses_cleaned_0.kml
[clean] all_warehouses_cleaned_2000.kml -> /content/kmls_cleaned/warehouses_cleaned_1.kml
[clean] all_warehouses_cleaned_4000.kml -> /content/kmls_cleaned/warehouses_cleaned_2.kml
[load] /content/kmls_cleaned/warehouses_cleaned_0.kml: rows = 2000
[load] /content/kmls_cleaned/warehouses_cleaned_1.kml: rows = 2000
[load] /content/kmls_cleaned/warehouses_cleaned_2.kml: rows = 351
[pos] unique cleaned point count = 4205
(4205, 17)


In [ ]:
#DBSCAN density labelling + parameter sweep
"""
Examined several DBSCAN parameter settings to assess whether local spatial
density patterns among positive warehouse locations were stable and interpretable.

eps=10 km / min_samples=5 provides the most balanced partition and is used as
the operational setting. Nearby settings serve as robustness checks.
"""
pos_geo = gpd.GeoDataFrame(
    pos_gdf[["pos_id", "lat", "lon"]].copy(),
    geometry=gpd.points_from_xy(pos_gdf["lon"], pos_gdf["lat"]),
    crs="EPSG:4326",
).to_crs(epsg=3857)
coords = np.column_stack([pos_geo.geometry.x, pos_geo.geometry.y])


def dbscan_summary(coords, eps_m, min_samples):
    labels = DBSCAN(eps=eps_m, min_samples=min_samples).fit_predict(coords)
    n_total = len(labels)
    n_noise = int(np.sum(labels == -1))
    non_noise = labels[labels != -1]
    n_clusters = len(set(non_noise)) if len(non_noise) > 0 else 0

    if n_clusters > 0:
        cluster_sizes = pd.Series(non_noise).value_counts().sort_values(ascending=False)
        largest_cluster         = int(cluster_sizes.iloc[0])
        median_cluster_size     = float(cluster_sizes.median())
        n_points_in_clusters    = int((labels != -1).sum())
        prop_points_in_clusters = n_points_in_clusters / n_total
        n_clusters_ge_20        = int((cluster_sizes >= 20).sum())
        prop_points_in_large    = cluster_sizes[cluster_sizes >= 20].sum() / n_total
    else:
        largest_cluster = median_cluster_size = n_points_in_clusters = 0
        prop_points_in_clusters = n_clusters_ge_20 = prop_points_in_large = 0.0

    return {
        "eps_km": eps_m / 1000, "min_samples": min_samples,
        "n_clusters": n_clusters, "n_noise": n_noise,
        "noise_prop": round(n_noise / n_total, 3),
        "largest_cluster": largest_cluster,
        "median_cluster_size": round(median_cluster_size, 1),
        "points_in_clusters": n_points_in_clusters,
        "prop_points_in_clusters": round(prop_points_in_clusters, 3),
        "n_clusters_ge_20": n_clusters_ge_20,
        "prop_points_in_large_clusters": round(prop_points_in_large, 3),
        "labels": labels,
    }


# --- Parameter sweep ---
eps_list_km        = [7.5, 10, 15]
min_samples_list   = [5, 8, 12]
sweep_results, labels_store = [], {}

for eps_km in eps_list_km:
    for ms in min_samples_list:
        out = dbscan_summary(coords, eps_m=eps_km * 1000, min_samples=ms)
        labels_store[(eps_km, ms)] = out["labels"]
        sweep_results.append({k: v for k, v in out.items() if k != "labels"})

sweep_df = pd.DataFrame(sweep_results).sort_values(["eps_km", "min_samples"]).reset_index(drop=True)
print("DBSCAN parameter sweep summary:")
print(sweep_df)
sweep_df.to_csv("/content/kmls_cleaned/dbscan_parameter_sweep_summary.csv", index=False)

# --- Apply chosen setting ---
CHOSEN_EPS_KM      = 10
CHOSEN_MIN_SAMPLES = 5
chosen_labels      = labels_store[(CHOSEN_EPS_KM, CHOSEN_MIN_SAMPLES)]

pos_dbscan = pos_geo.copy()
pos_dbscan["cluster_id"] = chosen_labels
cluster_size_map = (
    pos_dbscan[pos_dbscan["cluster_id"] != -1]
    .groupby("cluster_id").size().to_dict()
)
pos_dbscan["cluster_size"] = pos_dbscan["cluster_id"].map(cluster_size_map).fillna(0).astype(int)


def assign_density_type(row):
    if row["cluster_id"] == -1:         return "isolated"
    if row["cluster_size"] >= 20:       return "dense_cluster"
    return "small_cluster"


pos_dbscan["density_type"] = pos_dbscan.apply(assign_density_type, axis=1)
pos_dbscan = pos_dbscan.to_crs(epsg=4326)

pos_density = pos_dbscan[["pos_id", "lat", "lon", "cluster_id", "cluster_size", "density_type"]].copy()
print("\nChosen setting density label counts:")
print(pos_density["density_type"].value_counts())
print("\nSample density-labeled positives:")
print(pos_density.head())
pos_density.to_csv("/content/kmls_cleaned/positive_points_density_labels.csv", index=False)

DBSCAN parameter sweep summary:
   eps_km  min_samples  n_clusters  n_noise  noise_prop  largest_cluster  \
0     7.5            5         161     2423       0.576               88   
1     7.5            8          70     3089       0.735               59   
2     7.5           12          31     3554       0.845               56   
3    10.0            5         159     2070       0.492              113   
4    10.0            8          86     2628       0.625              106   
5    10.0           12          40     3210       0.763               93   
6    15.0            5         156     1507       0.358              156   
7    15.0            8          90     2045       0.486              143   
8    15.0           12          49     2620       0.623              140   

   median_cluster_size  points_in_clusters  prop_points_in_clusters  \
0                  7.0                1782                    0.424   
1                 11.5                1116                    0.2

In [ ]:
# 3.  Density-aware negative sampling (mini strategy study)

# NOTE: density_v3_conservative is excluded from this study.
#
# v3 sets dense_cluster search_dist=1200m but exclusion_buffer=1800m,
# meaning the exclusion zone is larger than the search radius. In practice,
# nearly every candidate found within the search circle is immediately
# filtered out, yielding almost zero negatives for dense-cluster points.
# The sampler then exhausts all seed points without reaching n_target,
# making runtime unpredictably long with no meaningful output.
#
# The three remaining strategies form a clean progression:
#   baseline_global  — flat params across all density types (control)
#   density_v1       — moderate differentiation by density
#   density_v2       — aggressive differentiation by density

pos_gdf_with_density = pos_gdf.merge(
    pos_density[["pos_id", "density_type", "cluster_id", "cluster_size"]],
    on="pos_id", how="left"
)
print("Density label counts:")
print(pos_gdf_with_density["density_type"].value_counts(dropna=False))


# --- Strategy grid ---
DENSITY_STRATEGY_GRID = {
    "baseline_global": {
        "isolated":      {"search_dist": 2500, "exclusion_buffer": 1000},
        "small_cluster": {"search_dist": 2500, "exclusion_buffer": 1000},
        "dense_cluster": {"search_dist": 2500, "exclusion_buffer": 1000},
    },
    "density_v1": {
        "isolated":      {"search_dist": 4000, "exclusion_buffer": 900},
        "small_cluster": {"search_dist": 2500, "exclusion_buffer": 1100},
        "dense_cluster": {"search_dist": 1500, "exclusion_buffer": 1500},
    },
    "density_v2": {
        "isolated":      {"search_dist": 5000, "exclusion_buffer": 800},
        "small_cluster": {"search_dist": 3000, "exclusion_buffer": 1200},
        "dense_cluster": {"search_dist": 1800, "exclusion_buffer": 1600},
    },
}

# --- All 10 negative categories ---
NEGATIVE_CATEGORIES = {
    "fuel":         {"amenity": "fuel"},
    "parking":      {"amenity": "parking"},
    "restaurant":   {"amenity": "restaurant"},
    "hospital":     {"amenity": "hospital"},
    "hotel":        {"tourism": "hotel"},
    "supermarket":  {"shop": "supermarket"},
    "school":       {"amenity": "school"},
    "park":         {"leisure": "park"},
    "playground":   {"leisure": "playground"},
    "sports_centre":{"leisure": "sports_centre"},
}

# --- Seed allocation ---
SEED_ALLOC = {"isolated": 20, "small_cluster": 15, "dense_cluster": 15}


# --- Helper functions ---
def build_positive_exclusion_union(pos_gdf_wgs84, buffer_m=1000):
    pos_m = pos_gdf_wgs84.to_crs(epsg=3857)
    union = unary_union(pos_m.geometry.buffer(buffer_m))
    print(f"[buffer] built union buffer with {buffer_m}m")
    return union


def to_point_geometry(geom):
    if geom is None or getattr(geom, "is_empty", False): return None
    if geom.geom_type == "Point": return geom
    try:    return geom.representative_point()
    except Exception:
        try: return geom.centroid
        except Exception: return None


def collect_local_category_samples(
    pos_gdf, pos_buffer_union, tags, category_name,
    n_target=100, n_seed=20, search_dist=1200,
    oversample_factor=1.3, random_state=42,
):
    """
    Local sampling: sample seed positive points, query OSM, remove points
    within exclusion buffer, dedupe, and sample down to n_target.
    """
    t0 = time.time()
    if len(pos_gdf) == 0:
        return None, {"category": category_name, "candidates": 0, "seconds": 0.0}

    seed_points   = pos_gdf.sample(n=min(n_seed, len(pos_gdf)), random_state=random_state).copy()
    collected, seen = [], set()
    target_collect  = int(n_target * oversample_factor)

    for _, seed in seed_points.iterrows():
        center = (float(seed["lat"]), float(seed["lon"]))
        try:
            gdf_cat = ox.features_from_point(center, tags=tags, dist=search_dist)
        except Exception as e:
            print(f"[{category_name}] OSM query failed near {center}: {e}")
            continue

        if gdf_cat is None or len(gdf_cat) == 0: continue

        gdf_cat = gdf_cat.reset_index()
        gdf_cat = gdf_cat[gdf_cat.geometry.notnull()].copy()
        gdf_cat["geometry"] = gdf_cat["geometry"].apply(to_point_geometry)
        gdf_cat = gdf_cat[gdf_cat.geometry.notnull()].copy()
        if len(gdf_cat) == 0: continue

        gdf_cat = gpd.GeoDataFrame(gdf_cat, geometry="geometry", crs="EPSG:4326")
        gdf_cat["lon"] = gdf_cat.geometry.x
        gdf_cat["lat"] = gdf_cat.geometry.y
        gdf_cat = gdf_cat.dropna(subset=["lat", "lon"]).copy()

        gdf_cat_m  = gdf_cat.to_crs(epsg=3857)
        gdf_cat_m  = gdf_cat_m[~gdf_cat_m.geometry.within(pos_buffer_union)].copy()
        if len(gdf_cat_m) == 0: continue

        gdf_cat = gdf_cat_m.to_crs(epsg=4326)
        gdf_cat["neg_category"] = category_name
        gdf_cat["coord_key"] = gdf_cat.apply(
            lambda r: (round(float(r["lat"]), 6), round(float(r["lon"]), 6)), axis=1
        )
        gdf_cat = gdf_cat[~gdf_cat["coord_key"].isin(seen)].copy()
        seen.update(gdf_cat["coord_key"].tolist())
        if len(gdf_cat) == 0: continue

        collected.append(gdf_cat[["geometry", "lat", "lon", "neg_category"]].copy())
        current = sum(len(x) for x in collected)
        print(f"[{category_name}] candidates so far: {current}")
        if current >= target_collect: break

    if len(collected) == 0:
        elapsed = time.time() - t0
        print(f"[{category_name}] no candidates found. (time {elapsed:.1f}s)")
        return None, {"category": category_name, "candidates": 0, "seconds": elapsed}

    out = pd.concat(collected, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs="EPSG:4326")
    out = out.drop_duplicates(subset=["lat", "lon"]).copy()
    out = out.sample(n=min(n_target, len(out)), random_state=random_state).reset_index(drop=True)

    elapsed = time.time() - t0
    print(f"[{category_name}] final kept: {len(out)} (time {elapsed:.1f}s)")
    return out, {"category": category_name, "candidates": len(out), "seconds": elapsed}


def collect_local_category_samples_by_density(
    pos_gdf_with_density, tags, category_name, density_strategy,
    n_target=100, seed_alloc=None, oversample_factor=1.08, random_state=42,
):
    t0 = time.time()
    if seed_alloc is None:
        seed_alloc = {"isolated": 20, "small_cluster": 15, "dense_cluster": 15}

    density_types      = ["isolated", "small_cluster", "dense_cluster"]
    collected_parts, diagnostics = [], []
    planned_total_seed = sum(seed_alloc.values())

    for density_type in density_types:
        pos_subset = pos_gdf_with_density[pos_gdf_with_density["density_type"] == density_type].copy()
        params          = density_strategy[density_type]
        search_dist     = params["search_dist"]
        exclusion_buffer = params["exclusion_buffer"]
        n_seed          = min(seed_alloc.get(density_type, 0), len(pos_subset))

        diag_base = {
            "category": category_name, "density_type": density_type,
            "n_pos_available": len(pos_subset), "n_seed_used": n_seed,
            "search_dist": search_dist, "exclusion_buffer": exclusion_buffer,
        }

        if len(pos_subset) == 0 or n_seed == 0:
            diagnostics.append({**diag_base, "n_candidates_kept": 0})
            continue

        pos_buffer_union = build_positive_exclusion_union(pos_gdf_with_density, buffer_m=exclusion_buffer)
        density_target   = max(1, int(round(n_target * (n_seed / planned_total_seed))))

        neg_part, _ = collect_local_category_samples(
            pos_gdf=pos_subset, pos_buffer_union=pos_buffer_union, tags=tags,
            category_name=f"{category_name}__{density_type}",
            n_target=density_target, n_seed=n_seed,
            search_dist=search_dist, oversample_factor=oversample_factor,
            random_state=random_state,
        )

        n_kept = 0
        if neg_part is not None and len(neg_part) > 0:
            neg_part = neg_part.copy()
            neg_part["seed_density_type"]    = density_type
            neg_part["search_dist_used"]     = search_dist
            neg_part["exclusion_buffer_used"] = exclusion_buffer
            neg_part["base_category"]        = category_name
            neg_part["neg_category"]         = category_name
            collected_parts.append(neg_part)
            n_kept = len(neg_part)

        diagnostics.append({**diag_base, "n_candidates_kept": n_kept})

    if len(collected_parts) == 0:
        elapsed = time.time() - t0
        return None, {"category": category_name, "seconds": elapsed, "final_kept": 0,
                      "diagnostics_df": pd.DataFrame(diagnostics)}

    out = pd.concat(collected_parts, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs="EPSG:4326")
    out["coord_key"] = out.apply(
        lambda r: (round(float(r["lat"]), 6), round(float(r["lon"]), 6)), axis=1
    )
    out = out.drop_duplicates(subset=["coord_key"]).drop(columns=["coord_key"])
    if len(out) > n_target:
        out = out.sample(n=n_target, random_state=random_state).copy()
    out = out.reset_index(drop=True)

    elapsed = time.time() - t0
    return out, {"category": category_name, "seconds": elapsed, "final_kept": len(out),
                 "diagnostics_df": pd.DataFrame(diagnostics)}


def run_density_strategy_mini_study(
    pos_gdf_with_density, negative_categories, strategy_grid,
    n_target_per_category=100, seed_alloc=None,
    oversample_factor=1.08, random_state=42,
):
    all_strategy_results = {}
    strategy_summary_rows, strategy_diag_rows = [], []

    for strategy_name, density_strategy in strategy_grid.items():
        print("\n" + "=" * 70)
        print(f"Running strategy: {strategy_name}")
        print("=" * 70)

        all_negs, per_cat_rows = [], []

        for cat, tags in negative_categories.items():
            print(f"\n[strategy={strategy_name}] category={cat}")
            neg_cat_gdf, out = collect_local_category_samples_by_density(
                pos_gdf_with_density=pos_gdf_with_density, tags=tags,
                category_name=cat, density_strategy=density_strategy,
                n_target=n_target_per_category, seed_alloc=seed_alloc,
                oversample_factor=oversample_factor, random_state=random_state,
            )

            diag_df = out["diagnostics_df"].copy()
            diag_df["strategy_name"]           = strategy_name
            diag_df["category_total_final_kept"] = out["final_kept"]
            diag_df["category_seconds"]          = out["seconds"]
            strategy_diag_rows.append(diag_df)

            per_cat_rows.append({
                "strategy_name": strategy_name, "category": cat,
                "final_kept": out["final_kept"], "seconds": out["seconds"],
            })

            if neg_cat_gdf is not None and len(neg_cat_gdf) > 0:
                neg_cat_gdf = neg_cat_gdf.copy()
                neg_cat_gdf["strategy_name"] = strategy_name
                all_negs.append(neg_cat_gdf)

        per_cat_df = pd.DataFrame(per_cat_rows)

        if len(all_negs) > 0:
            neg_gdf = gpd.GeoDataFrame(
                pd.concat(all_negs, ignore_index=True),
                geometry="geometry", crs="EPSG:4326"
            ).reset_index(drop=True)
            neg_gdf["sample_id"]  = neg_gdf.index.astype(str).str.zfill(7)
            neg_gdf["image_name"] = "neg_" + neg_gdf["sample_id"] + ".png"
        else:
            neg_gdf = None

        total_final = 0 if neg_gdf is None else len(neg_gdf)
        strategy_summary_rows.append({
            "strategy_name":         strategy_name,
            "n_categories":          len(per_cat_df),
            "total_final_negatives": total_final,
            "mean_per_category":     per_cat_df["final_kept"].mean()   if len(per_cat_df) else 0,
            "median_per_category":   per_cat_df["final_kept"].median() if len(per_cat_df) else 0,
            "total_runtime_sec":     per_cat_df["seconds"].sum()       if len(per_cat_df) else 0,
        })
        all_strategy_results[strategy_name] = {
            "neg_gdf": neg_gdf,
            "per_category_summary": per_cat_df,
        }

    strategy_summary_df = pd.DataFrame(strategy_summary_rows)
    strategy_diag_df    = pd.concat(strategy_diag_rows, ignore_index=True)
    return all_strategy_results, strategy_summary_df, strategy_diag_df


# --- Run mini study ---
all_strategy_results, strategy_summary_df, strategy_diag_df = run_density_strategy_mini_study(
    pos_gdf_with_density=pos_gdf_with_density,
    negative_categories=NEGATIVE_CATEGORIES,
    strategy_grid=DENSITY_STRATEGY_GRID,
    n_target_per_category=100,
    seed_alloc=SEED_ALLOC,
    oversample_factor=1.08,
    random_state=42,
)

print("\n=== Strategy summary ===")
print(strategy_summary_df.sort_values("total_final_negatives", ascending=False))
print("\n=== Diagnostics sample ===")
print(strategy_diag_df.head(20))

Density label counts:
density_type
isolated         2070
small_cluster    1138
dense_cluster     997
Name: count, dtype: int64

Running strategy: baseline_global

[strategy=baseline_global] category=fuel
[buffer] built union buffer with 1000m
[fuel__isolated] candidates so far: 12
[fuel__isolated] candidates so far: 14
[fuel__isolated] candidates so far: 16
[fuel__isolated] candidates so far: 19
[fuel__isolated] candidates so far: 21
[fuel__isolated] candidates so far: 26
[fuel__isolated] candidates so far: 31
[fuel__isolated] candidates so far: 40
[fuel__isolated] candidates so far: 43
[fuel__isolated] final kept: 40 (time 68.6s)
[buffer] built union buffer with 1000m
[fuel__small_cluster] candidates so far: 5
[fuel__small_cluster] candidates so far: 7
[fuel__small_cluster] candidates so far: 11
[fuel__small_cluster] candidates so far: 15
[fuel__small_cluster] candidates so far: 23
[fuel__small_cluster] candidates so far: 24
[fuel__small_cluster] candidates so far: 26
[fuel__small_clu

In [ ]:
#Saving results to csv files
for strategy_name, result in all_strategy_results.items():
    if result["neg_gdf"] is not None:
        result["neg_gdf"].drop(columns="geometry").to_csv(
            f"/content/kmls_cleaned/neg_{strategy_name}.csv", index=False
        )

strategy_summary_df.to_csv("/content/kmls_cleaned/strategy_summary.csv", index=False)
strategy_diag_df.to_csv("/content/kmls_cleaned/strategy_diag.csv", index=False)

In [ ]:
#Adding more iamges, runs only for deficient (strategy, category) pairs

#Load existing CSV
BASE_DIR = "/content/kmls_cleaned"

df_v2  = pd.read_csv(os.path.join(BASE_DIR, "neg_density_v2.csv"))
df_bg  = pd.read_csv(os.path.join(BASE_DIR, "neg_baseline_global.csv"))

print("Loaded density_v2:", len(df_v2))
print("Loaded baseline_global:", len(df_bg))

#Define what needs topping up
N_TARGET = 100

def compute_gaps(df, strategy_name):
    counts = df.groupby("neg_category").size()
    gaps = {}
    for cat, cnt in counts.items():
        if cnt < N_TARGET:
            gaps[cat] = N_TARGET - cnt
    return gaps

gaps_v2 = compute_gaps(df_v2, "density_v2")
gaps_bg = compute_gaps(df_bg, "baseline_global")

print("\nGaps in density_v2:",  gaps_v2)
print("Gaps in baseline_global:", gaps_bg)

#Expanded tag sets (hospital gets clinic + doctors)
TOPUP_TAGS = {
    "hospital":     [{"amenity": "hospital"}, {"amenity": "clinic"}, {"amenity": "doctors"}],
    "fuel":         [{"amenity": "fuel"}],
    "hotel":        [{"tourism": "hotel"}, {"tourism": "motel"}, {"tourism": "guest_house"}],
    "supermarket":  [{"shop": "supermarket"}, {"shop": "wholesale"}],
    "sports_centre":[{"leisure": "sports_centre"}, {"leisure": "stadium"}, {"leisure": "fitness_centre"}],
    "parking":      [{"amenity": "parking"}],
    "restaurant":   [{"amenity": "restaurant"}],
    "school":       [{"amenity": "school"}],
    "park":         [{"leisure": "park"}],
    "playground":   [{"leisure": "playground"}],
}

# Top-up sampling params per strategy
# Use slightly relaxed params vs. original to maximise yield
TOPUP_PARAMS = {
    "density_v2": {
        # use a flat generous setting for top-up queries
        "search_dist": 6000,
        "exclusion_buffer": 800,
        "n_seed": 40,
    },
    "baseline_global": {
        "search_dist": 4000,
        "exclusion_buffer": 1000,
        "n_seed": 40,
    },
}

# Helpers (reuse logic from original pipeline)
def to_point_geometry(geom):
    if geom is None or getattr(geom, "is_empty", False): return None
    if geom.geom_type == "Point": return geom
    try:    return geom.representative_point()
    except Exception:
        try: return geom.centroid
        except Exception: return None


def build_exclusion_union(pos_gdf_wgs84, buffer_m):
    pos_m = pos_gdf_wgs84.to_crs(epsg=3857)
    return unary_union(pos_m.geometry.buffer(buffer_m))


def query_osm_multi_tags(center, tag_list, dist):
    """Query OSM with multiple tag dicts, union results."""
    frames = []
    for tags in tag_list:
        try:
            gdf = ox.features_from_point(center, tags=tags, dist=dist)
            if gdf is not None and len(gdf) > 0:
                frames.append(gdf.reset_index())
        except Exception as e:
            print(f"  [OSM] tags={tags} near {center}: {e}")
    if not frames:
        return None
    out = pd.concat(frames, ignore_index=True)
    return out


def topup_category(
    existing_df,        # existing negatives for this category (filtered)
    pos_gdf_wgs84,      # all positive points (GeoDataFrame, EPSG:4326)
    category_name,
    tag_list,
    n_need,             # how many more we need
    search_dist,
    exclusion_buffer,
    n_seed,
    strategy_name,
    random_state=42,
):
    """
    Sample seed positives, query OSM with expanded tags, filter,
    deduplicate against existing negatives, return top-up rows.
    """
    rng = random.Random(random_state)
    np.random.seed(random_state)

    # Build exclusion union from ALL positives
    pos_buffer_union = build_exclusion_union(pos_gdf_wgs84, exclusion_buffer)

    # Existing coord keys to avoid duplicates
    existing_keys = set(
        zip(existing_df["lat"].round(6), existing_df["lon"].round(6))
    ) if len(existing_df) > 0 else set()

    seed_idx = list(pos_gdf_wgs84.index)
    rng.shuffle(seed_idx)
    seed_idx = seed_idx[:n_seed]

    collected, seen = [], set(existing_keys)

    for idx in seed_idx:
        row = pos_gdf_wgs84.loc[idx]
        center = (float(row["lat"]), float(row["lon"]))

        raw = query_osm_multi_tags(center, tag_list, dist=search_dist)
        if raw is None or len(raw) == 0:
            continue

        raw = raw[raw.geometry.notnull()].copy()
        raw["geometry"] = raw["geometry"].apply(to_point_geometry)
        raw = raw[raw.geometry.notnull()].copy()
        if len(raw) == 0:
            continue

        gdf = gpd.GeoDataFrame(raw, geometry="geometry", crs="EPSG:4326")
        gdf["lon"] = gdf.geometry.x
        gdf["lat"] = gdf.geometry.y
        gdf = gdf.dropna(subset=["lat", "lon"]).copy()

        # Filter inside exclusion buffer
        gdf_m = gdf.to_crs(epsg=3857)
        gdf_m = gdf_m[~gdf_m.geometry.within(pos_buffer_union)].copy()
        if len(gdf_m) == 0:
            continue
        gdf = gdf_m.to_crs(epsg=4326)
        gdf["lon"] = gdf.geometry.x
        gdf["lat"] = gdf.geometry.y

        # Deduplicate
        gdf["coord_key"] = list(zip(gdf["lat"].round(6), gdf["lon"].round(6)))
        gdf = gdf[~gdf["coord_key"].isin(seen)].copy()
        seen.update(gdf["coord_key"].tolist())
        if len(gdf) == 0:
            continue

        collected.append(gdf[["lat", "lon"]].copy())
        total = sum(len(x) for x in collected)
        print(f"  [{category_name}] candidates: {total}")
        if total >= n_need * 2:   # collect 2× then sample down
            break

    if not collected:
        print(f"  [{category_name}] no new candidates found")
        return pd.DataFrame()

    out = pd.concat(collected, ignore_index=True).drop_duplicates(["lat", "lon"])
    out = out.sample(n=min(n_need, len(out)), random_state=random_state).reset_index(drop=True)
    out["neg_category"]          = category_name
    out["base_category"]         = category_name
    out["strategy_name"]         = strategy_name
    # Mark top-up rows with a flat density type and the params used
    out["seed_density_type"]     = "topup"
    out["search_dist_used"]      = search_dist
    out["exclusion_buffer_used"] = exclusion_buffer
    print(f"  [{category_name}] top-up kept: {len(out)}")
    return out


# Load positive GeoDataFrame
pos_csv = os.path.join(BASE_DIR, "positive_points_cleaned_dedup.csv")
pos_raw = pd.read_csv(pos_csv)
pos_gdf_wgs84 = gpd.GeoDataFrame(
    pos_raw,
    geometry=gpd.points_from_xy(pos_raw["lon"], pos_raw["lat"]),
    crs="EPSG:4326",
)
print(f"\nLoaded {len(pos_gdf_wgs84)} positive points")

# Run top-ups
def run_topups(existing_df, gaps, strategy_name, pos_gdf_wgs84):
    params   = TOPUP_PARAMS[strategy_name]
    new_rows = []

    for cat, n_need in gaps.items():
        print(f"\n── Topping up [{strategy_name}] {cat} (need +{n_need}) ──")
        existing_cat = existing_df[existing_df["neg_category"] == cat].copy()
        tag_list     = TOPUP_TAGS.get(cat, [{"amenity": cat}])

        extra = topup_category(
            existing_df     = existing_cat,
            pos_gdf_wgs84   = pos_gdf_wgs84,
            category_name   = cat,
            tag_list        = tag_list,
            n_need          = n_need,
            search_dist     = params["search_dist"],
            exclusion_buffer= params["exclusion_buffer"],
            n_seed          = params["n_seed"],
            strategy_name   = strategy_name,
        )
        if len(extra) > 0:
            new_rows.append(extra)
        time.sleep(0.5)   # be polite to OSM

    if not new_rows:
        print(f"No top-ups produced for {strategy_name}")
        return existing_df

    # Merge and re-index
    combined = pd.concat([existing_df] + new_rows, ignore_index=True)
    combined = combined.drop_duplicates(subset=["lat", "lon"]).reset_index(drop=True)
    # Enforce N_TARGET cap per category
    combined = (
        combined.groupby("neg_category", group_keys=False)
        .apply(lambda g: g.head(N_TARGET))
        .reset_index(drop=True)
    )
    # Re-assign sample_id and image_name
    combined["sample_id"]  = combined.index.astype(str).str.zfill(7)
    combined["image_name"] = "neg_" + combined["sample_id"] + ".png"
    return combined


df_v2_updated = run_topups(df_v2, gaps_v2, "density_v2",      pos_gdf_wgs84)
df_bg_updated = run_topups(df_bg, gaps_bg, "baseline_global",  pos_gdf_wgs84)

# Save updated CSVs
out_v2 = os.path.join(BASE_DIR, "neg_density_v2_filled.csv")
out_bg = os.path.join(BASE_DIR, "neg_baseline_global_filled.csv")

df_v2_updated.to_csv(out_v2, index=False)
df_bg_updated.to_csv(out_bg, index=False)

print("\n=== Final counts ===")
print("density_v2:")
print(df_v2_updated.groupby("neg_category").size().sort_index())
print("\nbaseline_global:")
print(df_bg_updated.groupby("neg_category").size().sort_index())
print(f"\nSaved: {out_v2}")
print(f"Saved: {out_bg}")

Loaded density_v2: 911
Loaded baseline_global: 900

Gaps in density_v2: {'fuel': 1, 'hospital': 46, 'hotel': 3, 'sports_centre': 20, 'supermarket': 19}
Gaps in baseline_global: {'hospital': 69, 'hotel': 11, 'sports_centre': 20}

Loaded 4205 positive points

── Topping up [density_v2] fuel (need +1) ──
  [fuel] candidates: 59
  [fuel] top-up kept: 1

── Topping up [density_v2] hospital (need +46) ──
  [OSM] tags={'amenity': 'hospital'} near (43.6605722, -79.65575299999999): No matching features. Check query location, tags, and log.
  [hospital] candidates: 33
  [hospital] candidates: 42
  [OSM] tags={'amenity': 'hospital'} near (33.4325285, -88.55654779999999): No matching features. Check query location, tags, and log.
  [OSM] tags={'amenity': 'clinic'} near (33.4325285, -88.55654779999999): No matching features. Check query location, tags, and log.
  [OSM] tags={'amenity': 'doctors'} near (33.4325285, -88.55654779999999): No matching features. Check query location, tags, and log.
  [ho

/tmp/ipykernel_30329/3187250443.py:224: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(N_TARGET))



── Topping up [baseline_global] hospital (need +69) ──
  [OSM] tags={'amenity': 'hospital'} near (43.6605722, -79.65575299999999): No matching features. Check query location, tags, and log.
  [OSM] tags={'amenity': 'doctors'} near (43.6605722, -79.65575299999999): No matching features. Check query location, tags, and log.
  [hospital] candidates: 2
  [hospital] candidates: 6
  [OSM] tags={'amenity': 'hospital'} near (33.4325285, -88.55654779999999): No matching features. Check query location, tags, and log.
  [OSM] tags={'amenity': 'clinic'} near (33.4325285, -88.55654779999999): No matching features. Check query location, tags, and log.
  [OSM] tags={'amenity': 'doctors'} near (33.4325285, -88.55654779999999): No matching features. Check query location, tags, and log.
  [hospital] candidates: 19
  [OSM] tags={'amenity': 'hospital'} near (29.56187, -98.4201358): No matching features. Check query location, tags, and log.
  [OSM] tags={'amenity': 'doctors'} near (29.56187, -98.4201358):

/tmp/ipykernel_30329/3187250443.py:224: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(N_TARGET))


In [ ]:
# ── Hard Negative Sampling
N_TARGET_PER_CAT = 100   # hard neg per category (same as soft neg)
N_SEED           = 40
SEARCH_DIST      = 4000  # flat param — same for both strategies
EXCL_BUFFER      = 1000  # same exclusion buffer as baseline_global

HARD_NEG_TAGS = {
    "industrial":   [{"landuse": "industrial"}],
    "factory":      [{"building": "factory"}],
    "retail_large": [{"building": "retail"}],
    "commercial":   [{"building": "commercial"}],
    "wholesale":    [{"shop": "wholesale"}, {"shop": "doityourself"}],
}

# Load positives (needed as seed + exclusion source)
pos_csv = os.path.join(BASE_DIR, "positive_points_cleaned_dedup.csv")
pos_raw = pd.read_csv(pos_csv)
pos_gdf = gpd.GeoDataFrame(
    pos_raw,
    geometry=gpd.points_from_xy(pos_raw["lon"], pos_raw["lat"]),
    crs="EPSG:4326",
)

# ── Helpers ─────────────────────────────────────────────────────────────────
def to_point_geometry(geom):
    if geom is None or getattr(geom, "is_empty", False): return None
    if geom.geom_type == "Point": return geom
    try:    return geom.representative_point()
    except Exception:
        try: return geom.centroid
        except Exception: return None


def build_exclusion_union(pos_gdf_wgs84, buffer_m):
    pos_m = pos_gdf_wgs84.to_crs(epsg=3857)
    return unary_union(pos_m.geometry.buffer(buffer_m))


def query_hard_neg(category_name, tag_list, pos_gdf, n_target,
                   n_seed, search_dist, excl_buffer, random_state=42):
    rng = random.Random(random_state)
    np.random.seed(random_state)

    pos_buffer_union = build_exclusion_union(pos_gdf, excl_buffer)

    seed_idx = list(pos_gdf.index)
    rng.shuffle(seed_idx)
    seed_idx = seed_idx[:n_seed]

    collected, seen = [], set()

    for idx in seed_idx:
        row    = pos_gdf.loc[idx]
        center = (float(row["lat"]), float(row["lon"]))

        frames = []
        for tags in tag_list:
            try:
                gdf = ox.features_from_point(center, tags=tags, dist=search_dist)
                if gdf is not None and len(gdf) > 0:
                    frames.append(gdf.reset_index())
            except Exception as e:
                print(f"  [OSM] {category_name} tags={tags}: {e}")

        if not frames: continue
        raw = pd.concat(frames, ignore_index=True)

        raw = raw[raw.geometry.notnull()].copy()
        raw["geometry"] = raw["geometry"].apply(to_point_geometry)
        raw = raw[raw.geometry.notnull()].copy()
        if len(raw) == 0: continue

        gdf = gpd.GeoDataFrame(raw, geometry="geometry", crs="EPSG:4326")
        gdf["lon"] = gdf.geometry.x
        gdf["lat"] = gdf.geometry.y
        gdf = gdf.dropna(subset=["lat", "lon"]).copy()

        gdf_m = gdf.to_crs(epsg=3857)
        gdf_m = gdf_m[~gdf_m.geometry.within(pos_buffer_union)].copy()
        if len(gdf_m) == 0: continue
        gdf = gdf_m.to_crs(epsg=4326)
        gdf["lon"] = gdf.geometry.x
        gdf["lat"] = gdf.geometry.y

        gdf["coord_key"] = list(zip(gdf["lat"].round(6), gdf["lon"].round(6)))
        gdf = gdf[~gdf["coord_key"].isin(seen)].copy()
        seen.update(gdf["coord_key"].tolist())
        if len(gdf) == 0: continue

        collected.append(gdf[["lat", "lon"]].copy())
        total = sum(len(x) for x in collected)
        print(f"  [{category_name}] candidates: {total}")
        if total >= n_target * 2: break

    if not collected:
        print(f"  [{category_name}] no candidates found")
        return pd.DataFrame()

    out = pd.concat(collected, ignore_index=True).drop_duplicates(["lat", "lon"])
    out = out.sample(n=min(n_target, len(out)), random_state=random_state).reset_index(drop=True)
    print(f"  [{category_name}] kept: {len(out)}")
    return out


# ── Run queries ──────────────────────────────────────────────────────────────
print("\n— Querying hard negatives from OSM ————————————————————")
hard_neg_parts = []

for cat_name, tag_list in HARD_NEG_TAGS.items():
    print(f"\n  Category: {cat_name}")
    result = query_hard_neg(
        category_name = cat_name,
        tag_list      = tag_list,
        pos_gdf       = pos_gdf,
        n_target      = N_TARGET_PER_CAT,
        n_seed        = N_SEED,
        search_dist   = SEARCH_DIST,
        excl_buffer   = EXCL_BUFFER,
        random_state  = RANDOM_STATE,
    )
    if len(result) > 0:
        result["neg_category"]  = cat_name
        result["base_category"] = cat_name
        hard_neg_parts.append(result)
    time.sleep(1)


# ── Append hard negs to both strategy CSVs ───────────────────────────────────

if not hard_neg_parts:
    print("No hard negatives found — check OSM connectivity")
else:
    hard_neg_df = pd.concat(hard_neg_parts, ignore_index=True)
    print(f"\nTotal hard negatives collected: {len(hard_neg_df)}")
    print(hard_neg_df["neg_category"].value_counts())

    for strategy_name, csv_path in [
        ("density_v2",      os.path.join(BASE_DIR, "neg_density_v2_filled.csv")),
        ("baseline_global", os.path.join(BASE_DIR, "neg_baseline_global_filled.csv")),
    ]:
        existing = pd.read_csv(csv_path)

        # Tag hard neg rows
        hn = hard_neg_df.copy()
        hn["strategy_name"]         = strategy_name
        hn["seed_density_type"]     = "hard_negative"
        hn["search_dist_used"]      = SEARCH_DIST
        hn["exclusion_buffer_used"] = EXCL_BUFFER

        # Deduplicate against existing coords
        existing_keys = set(zip(existing["lat"].round(6), existing["lon"].round(6)))
        hn["coord_key"] = list(zip(hn["lat"].round(6), hn["lon"].round(6)))
        hn = hn[~hn["coord_key"].isin(existing_keys)].drop(columns=["coord_key"])

        combined = pd.concat([existing, hn], ignore_index=True)
        combined["sample_id"]  = combined.index.astype(str).str.zfill(7)
        combined["image_name"] = "neg_" + combined["sample_id"] + ".png"
        combined.to_csv(csv_path, index=False)

        print(f"\n[{strategy_name}] {len(existing)} → {len(combined)} (+{len(hn)} hard neg)")
        print(combined["neg_category"].value_counts().to_string())

    print("\nHard negatives appended to both CSVs.")
    print("Next: re-run YOLO prep cell with same paths — hard neg included automatically.")

In [ ]:
from google.colab import drive
drive.mount('/drive')

import shutil, os

#saving all output for futuer use
for folder in ["kmls_cleaned", "dataset_build"]:
    src = f"/content/{folder}"
    dst = f"/drive/MyDrive/{folder}"
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Saved: {src} → {dst}")

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


KeyboardInterrupt: 